In [1]:
!pip install tensorflow scikit-learn numpy pandas

In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

In [3]:
df = pd.read_csv("/content/GNSS_Merged_Dataset.csv")

print(df.shape)
print(df.head())

(6000, 25)
   Time(ms)    Temp(C)  Humidity(%)  RainAnalog  RainDigital  SNR1  SNR2  \
0    178200  33.691820    40.216222          60            0    49    50   
1    191700  25.062289    65.934869         317            1    42    41   
2     22100  34.546469    50.813970          61            0    52    49   
3     13500  30.950958    63.467639         440            1    41    39   
4    195000  22.264200    96.614697         854            1    35    34   

   SNR3  SNR4  SNR5  ...  SNR11  SNR12  SNR13  SNR14  SNR15   Mean_SNR  \
0    49    50    49  ...    NaN    NaN    NaN    NaN    NaN  49.428571   
1    43    39    42  ...    NaN    NaN    NaN    NaN    NaN  42.125000   
2    48    52    50  ...    NaN    NaN    NaN    NaN    NaN  50.428571   
3    39    43    39  ...    NaN    NaN    NaN    NaN    NaN  41.125000   
4    33    32    33  ...    NaN    NaN    NaN    NaN    NaN  31.000000   

     SNR_Var  Sat_Count  SNR_Diff  WeatherLabel  
0   0.285714          7 -0.142857    

In [4]:
feature_columns = [f"SNR{i}" for i in range(1,16)]

X = df[feature_columns].values
y = df["WeatherLabel"].values

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [6]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


In [7]:
mean = scaler.mean_
std = scaler.scale_

np.save("scaler_mean.npy", mean)
np.save("scaler_std.npy", std)

print("Mean values:\n", mean)
print("Std values:\n", std)

Mean values:
 [41.3375     41.33666667 41.38270833 41.27833333 41.37666667 41.271875
 41.48854167 36.403125   33.340625           nan         nan         nan
         nan         nan         nan]
Std values:
 [7.08641967 7.15180552 7.04571331 7.10270117 7.14567157 7.24169356
 7.09335619 5.32353174 2.85737635        nan        nan        nan
        nan        nan        nan]


In [8]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(12, activation="relu", input_shape=(15,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(3, activation="softmax")
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [10]:
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1
)

Epoch 1/50
135/135 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.3335 - loss: nan - val_accuracy: 0.3354 - val_loss: nan
Epoch 2/50
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.3265 - loss: nan - val_accuracy: 0.3354 - val_loss: nan
Epoch 3/50
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.3402 - loss: nan - val_accuracy: 0.3354 - val_loss: nan
Epoch 4/50
135/135 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3274 - loss: nan - val_accuracy: 0.3354 - val_loss: nan
Epoch 5/50
135/135 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3393 - loss: nan - val_accuracy: 0.3354 - val_loss: nan
Epoch 6/50
135/135 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3318 - loss: nan - val_accuracy: 0.3354 - val_loss: nan
Epoch 7/50
135/135 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3377 - loss: nan - val_accuracy: 0.3354 - val_loss: nan
Epoch 8/50
135/135 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3378 - loss: nan - val_accuracy: 0.3354 - val_loss: nan
Epoch 9/50
135/135 ━━━━━

In [11]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)
y_pred = np.argmax(model.predict(X_test), axis=1)
print(classification_report(y_test, y_pred))

38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3485 - loss: nan
Test Accuracy: 0.3333333432674408
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
              precision    recall  f1-score   support

           0       0.33      1.00      0.50       400
           1       0.00      0.00      0.00       400
           2       0.00      0.00      0.00       400

    accuracy                           0.33      1200
   macro avg       0.11      0.33      0.17      1200
weighted avg       0.11      0.33      0.17      1200



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [12]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open("rain_model.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model saved successfully.")

Saved artifact at '/tmp/tmptfy5i__p'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 15), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  136554712382160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136554712382928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136554712380240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136554712380432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136554712381200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136554712381968: TensorSpec(shape=(), dtype=tf.resource, name=None)
TFLite model saved successfully.


In [15]:
!apt-get install xxd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
xxd is already the newest version (2:8.2.3995-1ubuntu2.24).
xxd set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [16]:
!xxd -i rain_model.tflite > rain_model.cc

In [17]:
!ls -lh rain_model.cc

-rw-r--r-- 1 root root 21K Feb 13 16:50 rain_model.cc
